In [ ]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans 
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
protein = "VCAM1"
synapse_type = "VGAT-GEPH"

In [ ]:
vgat_geph_results_file = f"/Volumes/Intenso/image-analysis/synapse-counting/{protein}/{protein}-LacZ_VGAT-GEPH_output_data/metric_results.csv"
vgat_geph_results = pd.read_csv(vgat_geph_results_file)
vgat_geph_results

In [ ]:
# creating a new column with the sample_ID
vgat_geph_results["sample_ID"] = vgat_geph_results["gRNA"].str.cat(vgat_geph_results[["Brain"]], sep = "_")

In [ ]:
# checking specific feature
vgat_geph_results[(vgat_geph_results["Brain"] == "Brain-4") & (vgat_geph_results["hippocampal_layer"] == "DG ML") & (vgat_geph_results["gRNA"] == "LacZ-gRNA")]

In [ ]:
df2_grouped_brain = vgat_geph_results.groupby(["sample_ID", "hippocampal_layer"], as_index=False)[["local_peak_colocalized_spots",
                                        "overlap_coeff",
                                        "overlap_um2",
                                        "pearson_cor",
                                        "presynapse_image_mfi",
                                        "postsynapse_image_mfi",
                                        "pre_puncta_density_per_100_um2",
                                        "post_puncta_density_per_100_um2",
                                        "pre_staining_area_um2",
                                        "post_staining_area_um2",
                                        "pre_mean_puncta_size_um2",
                                        "post_mean_puncta_size_um2"]].mean()

In [ ]:
df2_grouped_brain

In [ ]:
df_grouped_section = vgat_geph_results[["sample_ID",
                                        "hippocampal_layer",
                                        "img_filename",
                                        "local_peak_colocalized_spots",
                                        "overlap_coeff",
                                        "overlap_um2",
                                        "pearson_cor",
                                        "presynapse_image_mfi",
                                        "postsynapse_image_mfi",
                                        "pre_puncta_density_per_100_um2",
                                        "post_puncta_density_per_100_um2",
                                        "pre_staining_area_um2",
                                        "post_staining_area_um2",
                                        "pre_mean_puncta_size_um2",
                                        "post_mean_puncta_size_um2"]]
final_df = df_grouped_section.reset_index()

In [ ]:
df_grouped_section

In [ ]:
# to long format
df_melted = df2_grouped_brain.melt(id_vars=['sample_ID', 'hippocampal_layer'], var_name='metric', value_name='value')

# create new column based on hippocampal layer and metric
df_melted['synapse_hippocampal_layer_metric'] = synapse_type + "_" + df_melted['hippocampal_layer'] + '_' + df_melted['metric']

# pivot the dataframe back
df_pivot = df_melted.pivot(index='sample_ID', columns='synapse_hippocampal_layer_metric', values='value')

vgat_final_df = df_pivot.reset_index()
vgat_final_df.head(10)

In [ ]:
protein = "VCAM1"
synapse_type = "VGLUT1-PSD95"

vglut1_psd95_results_file = f"/Volumes/Intenso/image-analysis/synapse-counting/{protein}/{protein}-LacZ_VGLUT1-PSD95_output_data/metric_results.csv"
vglut1_psd95_results = pd.read_csv(vglut1_psd95_results_file)
vglut1_psd95_results

In [ ]:
# creating a new column with the sample_ID
vglut1_psd95_results["sample_ID"] = vglut1_psd95_results["gRNA"].str.cat(vglut1_psd95_results[["Brain"]], sep = "_")

In [ ]:
df2_grouped_brain = vglut1_psd95_results.groupby(["sample_ID", "hippocampal_layer"], as_index=False)[["local_peak_colocalized_spots",
                                        "overlap_coeff",
                                        "overlap_um2",
                                        "pearson_cor",
                                        "presynapse_image_mfi",
                                        "postsynapse_image_mfi",
                                        "pre_puncta_density_per_100_um2",
                                        "post_puncta_density_per_100_um2",
                                        "pre_staining_area_um2",
                                        "post_staining_area_um2",
                                        "pre_mean_puncta_size_um2",
                                        "post_mean_puncta_size_um2"]].mean()

In [ ]:
# to long format
df_melted = df2_grouped_brain.melt(id_vars=['sample_ID', 'hippocampal_layer'], var_name='metric', value_name='value')

# create new column based on hippocampal layer and metric
df_melted['synapse_hippocampal_layer_metric'] = synapse_type + "_" + df_melted['hippocampal_layer'] + '_' + df_melted['metric']

# pivot the dataframe back
df_pivot = df_melted.pivot(index='sample_ID', columns='synapse_hippocampal_layer_metric', values='value')

vglut1_final_df = df_pivot.reset_index()
vglut1_final_df.head(10)

In [ ]:
# joining the VGAT & VGLUT1 data
merged_df = pd.merge(vgat_final_df, vglut1_final_df, on = "sample_ID", how = "outer")
merged_df_index = merged_df.set_index("sample_ID")
merged_df_index_log2 = np.log2(merged_df_index)
merged_df_index_log2

In [ ]:
# creating metadata
metadata = merged_df[["sample_ID"]].rename_axis(None, axis=1)
metadata[["Hemisphere", "Brain"]] = metadata["sample_ID"].str.split('_', expand= True)
metadata.set_index("sample_ID")

In [ ]:
data_with_meta = pd.merge(merged_df_index_log2, metadata, on="sample_ID", how="left")
data_with_meta

In [ ]:
feature_cols_only = data_with_meta.columns.drop(["sample_ID", "Brain", "Hemisphere"])
brain_means = data_with_meta.groupby("Brain")[feature_cols_only].mean()

brain_means

In [ ]:
adjusted_data_list = []
for brain in metadata["Brain"].unique():
    # Get samples belonging to this brain
    brain_samples = data_with_meta[data_with_meta["Brain"] == brain].copy()

    # Get the mean for this specific brain
    current_brain_mean = brain_means.loc[brain]

    # Subtract the brain mean from the feature values for these samples
    # We use .loc to ensure alignment by feature column names
    adjusted_samples = brain_samples[feature_cols_only] - current_brain_mean

    # Reattach SampleID, BrainID, Hemisphere for plotting later
    adjusted_samples["sample_ID"] = brain_samples["sample_ID"]
    adjusted_samples["Brain"] = brain_samples["Brain"]
    adjusted_samples["Hemisphere"] = brain_samples["Hemisphere"]
    adjusted_data_list.append(adjusted_samples)

# Concatenate all adjusted dataframes
adjusted_data_df = pd.concat(adjusted_data_list).set_index("sample_ID")

# Drop the metadata columns from the final adjusted_data_df to get just features
adjusted_features_df = adjusted_data_df.drop(columns=["Brain", "Hemisphere"])


adjusted_features_df

In [ ]:
# PCA analysis
scaler = StandardScaler()
scaled_adjusted_features = scaler.fit_transform(adjusted_features_df)

pca = PCA()
principal_components = pca.fit_transform(scaled_adjusted_features)

print(principal_components)

In [ ]:
pc_df = pd.DataFrame(data=principal_components,
                     columns=[f'PC{i+1}' for i in range(principal_components.shape[1])],
                     index=adjusted_data_df.index)
pc_df

In [ ]:
pc_df_with_metadata = pd.merge(pc_df, metadata, on = "sample_ID", how = "outer")
pc_df_with_metadata

In [ ]:
import matplotlib.pyplot as plt

# Plot explained variance ratio
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(pca.explained_variance_ratio_) + 1), pca.explained_variance_ratio_, marker='o')
plt.title('Scree Plot (Explained Variance per PC)')
plt.xlabel('Principal Component Number')
plt.ylabel('Proportion of Variance Explained')
plt.xticks(range(1, len(pca.explained_variance_ratio_) + 1))
plt.grid(False)
plt.show()

In [ ]:
from psynlig import pca_explained_variance_bar
pca_explained_variance_bar(pca, alpha=0.8)

plt.show()

In [ ]:
import seaborn as sns

plt.figure(figsize=(8, 6))
sns.scatterplot(x='PC1', y='PC2', hue='Hemisphere', data=pc_df_with_metadata, s=100,
                palette={'LacZ-gRNA': 'grey', 'VCAM1-gRNA': 'blue'})
plt.title('PCA Plot (PC1 vs PC2)')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.grid(False)
plt.show()

# If you want to connect paired samples:
plt.figure(figsize=(7, 6))
sns.scatterplot(x='PC1', y='PC2', hue='Hemisphere', data=pc_df_with_metadata, s=100,
                palette={'LacZ-gRNA': 'grey', 'VCAM1-gRNA': 'blue'}, legend='full')
for pair_id in pc_df_with_metadata['Brain'].unique():
    pair_data = pc_df_with_metadata[pc_df_with_metadata['Brain'] == pair_id]
    if len(pair_data) == 2: # Ensure both WT and KO of the pair are present
        plt.plot(pair_data['PC1'], pair_data['PC2'], color='gray', linestyle='--', alpha=0.6)
plt.title('PCA Plot (PC1 vs PC2) with Paired Samples')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.grid(False)
plt.show()

In [ ]:
#### Pearson's correlation between metrics and PCs

from scipy.stats import pearsonr

feature_pc_correlations = pd.DataFrame(index=adjusted_features_df.columns, columns=pc_df.columns)
feature_pc_pvalues = pd.DataFrame(index=adjusted_features_df.columns, columns=pc_df.columns)


for feature in adjusted_features_df.columns:
    for pc_col in pc_df.columns:
        # Use pearsonr from scipy.stats which returns (correlation, p_value)
        # Check if there's enough variation to calculate correlation
        if adjusted_features_df[feature].nunique() > 1 and pc_df[pc_col].nunique() > 1:
            corr_val, p_val = pearsonr(adjusted_features_df[feature], pc_df[pc_col])
            feature_pc_correlations.loc[feature, pc_col] = corr_val
            feature_pc_pvalues.loc[feature, pc_col] = p_val
        else:
            # Assign NaN if correlation cannot be calculated (e.g., no variance)
            feature_pc_correlations.loc[feature, pc_col] = np.nan
            feature_pc_pvalues.loc[feature, pc_col] = np.nan

feature_pc_correlations = feature_pc_correlations.astype(float)
feature_pc_pvalues = feature_pc_pvalues.astype(float)

feature_pc_correlations = feature_pc_correlations.drop(columns=["PC5", "PC6", "PC7", "PC8"])

feature_pc_pvalues = feature_pc_pvalues.drop(columns=["PC5", "PC6", "PC7", "PC8"])


In [ ]:
n_components_to_consider = 4

n_features = 192

pcs_for_sorting = ["PC1", "PC2", "PC3", "PC4"]

feature_pc_correlations['Abs_Sum'] = feature_pc_correlations.loc[:, pcs_for_sorting].abs().sum(axis=1)

sorted_features_by_custom_sum = feature_pc_correlations.sort_values(by='Abs_Sum', ascending=False).index

feature_pc_correlations = feature_pc_correlations.drop(columns=['Abs_Sum'])


feature_pc_correlations_sorted = feature_pc_correlations.loc[sorted_features_by_custom_sum]
feature_pc_pvalues_sorted = feature_pc_pvalues.loc[sorted_features_by_custom_sum]

alpha_level_1 = 0.05
alpha_level_2 = 0.01

annotation_matrix = feature_pc_pvalues_sorted.applymap(
    lambda x: '**' if x < alpha_level_2 else ('*' if x < alpha_level_1 else '')
)

# Add these lines BEFORE the sns.heatmap call
print(f"Shape of feature_pc_correlations_sorted: {feature_pc_correlations_sorted.shape}")
print(f"Shape of feature_pc_pvalues_sorted: {feature_pc_pvalues_sorted.shape}") # This is the source for annotation_matrix
print(f"Shape of annotation_matrix: {annotation_matrix.shape}") # To confirm annotation_matrix also has same shape

# Check if indices and columns are identical
print(f"Correlations sorted features (index) match P-values sorted features (index): {all(feature_pc_correlations_sorted.index == feature_pc_pvalues_sorted.index)}")
print(f"Correlations sorted PCs (columns) match P-values sorted PCs (columns): {all(feature_pc_correlations_sorted.columns == feature_pc_pvalues_sorted.columns)}")


plt.figure(figsize=(6 + n_components_to_consider * 0.8, n_features * 0.1))
sns.heatmap(feature_pc_correlations_sorted, cmap='coolwarm', center=0,
            linewidths=0.1, linecolor='lightgrey',
            cbar_kws={'label': 'Pearson Correlation Coefficient'},
            annot=annotation_matrix, fmt='', # Use empty format string to just show the '*' or '**'
            annot_kws={"size": 10, "color": "black"}) # Adjust annotation font size/color
plt.title(f'Pearson Correlation between Original Features and Top {n_components_to_consider} PCs\n'
          f'(Features Sorted by Absolute Sum of Correlations with {", ".join(pcs_for_sorting)})'
          f'\nSignificance: * p < {alpha_level_1}, ** p < {alpha_level_2}')
plt.xlabel('Principal Component')
plt.ylabel('Original Feature')
plt.yticks(rotation=0, fontsize=6)
plt.tight_layout()
plt.show()

In [ ]:
feature_pc_correlations_sorted

In [ ]:
g = sns.clustermap(
    feature_pc_correlations_sorted,
    row_cluster=True,
    col_cluster=False,
    cmap='RdBu_r',
    center=0,
    annot=False,  # Manual annotation comes next
    fmt=".2f",
    linewidths=0.5,
    cbar_kws={'label': 'Correlation between features & PCs'},
    figsize=(10, 25)
)

g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=4)
ax = g.ax_heatmap
ax.set_yticks(np.arange(len(g.data2d.index)) + 0.5)
ax.set_yticklabels(g.data2d.index, fontsize=8)  # Set desired font size

g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(), fontsize=9, rotation=0)

for i, feature in enumerate(g.data2d.index):
    for j, pc in enumerate(g.data2d.columns):
        text = annotation_matrix.loc[feature, pc]
        if text:  # Only annotate if there's something to show (not NaN or empty)
            g.ax_heatmap.text(j + 0.5, i + 0.5, text,
                              ha='center', va='center',
                              color='black', fontsize=10)

plt.show()


In [ ]:
sns.clustermap(
    feature_pc_correlations_sorted,
    row_cluster=True,  # Cluster features (rows)
    col_cluster=False, # Do not cluster PCs (columns) - keep them in order PC1, PC2, etc.
    cmap='RdBu_r',
    center=0,
    annot=False,
    fmt=".2f",
    linewidths=.5,
    cbar_kws={'label': 'Correlation between metrics & PCs'},
    figsize=(10, 12) # Adjust size
)

In [ ]:
feature_pc_correlations_sorted

In [ ]:
##### Pearsons correlation between metrics
from psynlig import plot_correlation_heatmap

kwargs = {
    'text': {
        'fontsize': 'small',
    },
    'heatmap': {
        'vmin': -1,
        'vmax': 1,
        'cmap': 'magma',
    },
    'figure': {'figsize': (100, 100)},
}

plot_correlation_heatmap(adjusted_features_df, textcolors=['white', 'black'], **kwargs)
plt.show()

In [ ]:
#### Pearsons correlation between PCs and features




In [ ]:
#### PCA loadings

loadings_matrix = pca.components_
loadings_df = pd.DataFrame(loadings_matrix.T,
                           index=adjusted_features_df.columns, # Original feature names
                           columns=[f'PC{i+1}' for i in range(pca.n_components_)])

print("Loadings DataFrame (head):")
print(loadings_df.head())

In [ ]:
# Sort features by their absolute loading on PC1
top_pc1_features = loadings_df['PC1'].abs().sort_values(ascending=False)
print("\nTop 10 features contributing to PC1:")
print(top_pc1_features.head(10))

# Get the actual loading values for these top features
print("\nTop 10 PC1 features with their loading values:")
print(loadings_df.loc[top_pc1_features.head(10).index, 'PC1'])


# Sort features by their absolute loading on PC2
top_pc2_features = loadings_df['PC2'].abs().sort_values(ascending=False)
print("\nTop 10 features contributing to PC2:")
print(top_pc2_features.head(10))

In [ ]:
loadings_df['Abs_PC_Contribution'] = loadings_df[["PC1", "PC2", "PC3"]].abs().sum(axis=1) # Sum of absolute loadings for PC1 and PC2

In [ ]:
top_n_features_for_heatmap = 25 # You want the top 20
top_features_loadings = loadings_df.sort_values(by='Abs_PC_Contribution', ascending=False).head(top_n_features_for_heatmap)


In [ ]:
heatmap_loadings_data = top_features_loadings.drop(columns=["PC4", "PC5", "PC6", "PC7", "PC8"])

print(f"\nHeatmap data (loadings for top {top_n_features_for_heatmap} features):")
print(heatmap_loadings_data)

# --- Create the Heatmap of Loadings ---
plt.figure(figsize=(10, 10)) # Adjust figure size for better readability

sns.heatmap(
    heatmap_loadings_data,
    cmap='RdBu_r', # Red-Blue diverging colormap (red for positive, blue for negative)
                   # 'vlag' is another good diverging option
    center=0,      # Ensure 0 is the white/neutral color
    annot=False,    # Show numerical values in cells (can make it crowded if too many features)
    fmt=".2f",     # Format annotation to 2 decimal places
    linewidths=.5, # Add lines between cells
    cbar_kws={'label': 'PCA Loading'} # Label for the color bar
)

plt.title(f'PCA Loadings Heatmap (Top {top_n_features_for_heatmap} Features)')
# plt.title(f'PCA Loadings Heatmap all Features')
plt.xlabel('Principal Component')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 10))
sns.clustermap(
    heatmap_loadings_data,
    row_cluster=True,  # Cluster features (rows)
    col_cluster=False, # Do not cluster PCs (columns) - keep them in order PC1, PC2, etc.
    cmap='RdBu_r',
    center=0,
    annot=True,
    fmt=".2f",
    linewidths=.5,
    cbar_kws={'label': 'PCA Loading'},
    figsize=(10, 12) # Adjust size
)
plt.suptitle(f'Clustered PCA Loadings Heatmap (Top {top_n_features_for_heatmap} Features)', y=1.02, fontsize=16)
plt.show()

In [ ]:
heatmap_data = adjusted_features_df

sample_hemispheres = [idx.split('_')[0] for idx in heatmap_data.index]
sample_brain_ids = [idx.split('_')[1] for idx in heatmap_data.index] # This will be 'Brain-4', 'Brain-4-2', etc.

# Create color mappings
# For Hemisphere
hemisphere_color_map = {'LacZ-gRNA': 'lightgrey', 'VCAM1-gRNA': 'skyblue'}
row_colors_hemisphere = [hemisphere_color_map[h] for h in sample_hemispheres]

# For BrainID (you have 'Brain-4', 'Brain-4-2', 'Brain-5', 'Brain-7')
unique_brain_ids = sorted(list(set(sample_brain_ids)))
brain_palette = sns.color_palette("tab10", len(unique_brain_ids))
brain_color_map = {brain_id_str: brain_palette[i] for i, brain_id_str in enumerate(unique_brain_ids)}
row_colors_brain = [brain_color_map[b] for b in sample_brain_ids]

# Combine into a DataFrame for clustermap's row_colors
row_colors_df = pd.DataFrame({
    'Hemisphere': row_colors_hemisphere,
    'BrainID': row_colors_brain
}, index=heatmap_data.index)


# --- Create the Heatmap ---
plt.figure(figsize=(12, 10)) # Adjust size as needed

sns.clustermap(
    heatmap_data,
    method='ward',      # Linkage method for clustering rows/columns
    metric='euclidean', # Distance metric for clustering
    z_score=0,          # Standardize features (columns) to z-scores (0 means scaling features across rows)
                        # z_score=1 would scale samples across columns
    cmap='vlag',        # Colormap for diverging data (e.g., values around 0)
    center=0,           # Center the colormap at 0
    row_colors=row_colors_df, # Annotate rows with colors for Hemisphere and BrainID
    col_cluster=True,   # Cluster columns (features)
    row_cluster=True,   # Cluster rows (samples)
    dendrogram_ratio=(.1, .2), # Adjust dendrogram size
    cbar_pos=(.95, .8, .02, .18), # Position of the color bar [left, bottom, width, height]
    figsize=(14, 12) # Overall figure size
)

plt.suptitle('Heatmap of Brain-Adjusted Features', y=1.02, fontsize=16) # Title for the figure
plt.show()

In [ ]:
heatmap_data.index